# Final Results: Random Forest, GraphSAGE, and Ensemble

This notebook presents **presentation-ready** tables and charts for the **final model comparison** using **stratified 5-fold cross-validation** on the **400** labelled infrastructure nodes, with **33** node features per graph (tabular + graph-derived structural features).

**Important labels:**

- All **main benchmark numbers** below are **5-fold CV means** (± standard deviation across folds where shown). They are **not** single train/validation/test split results.
- The **ensemble** is the **best overall cross-validated classification model** in this study (highest CV mean F1 among models compared).
- **GraphSAGE** is the **strongest individual graph model** (highest CV mean ROC-AUC among single models).

**Ensemble definition:** `ensemble_prob = 0.6 * rf_prob + 0.4 * gnn_prob`; `ensemble_pred = 1` if `ensemble_prob >= 0.5` else `0`.

Data source: `outputs/ensemble_cv_results.json` (produced by `src/ensemble_cv.py`).

In [ ]:
import json
import pathlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

ROOT = pathlib.Path("..").resolve()
OUTPUTS = ROOT / "outputs"

with open(OUTPUTS / "ensemble_cv_results.json") as f:
    ENS = json.load(f)

BEST_ALPHA = ENS["_best_ensemble"]["alpha"]
BEST_TAG = ENS["_best_ensemble"]["model_key"]
print(f"Loaded ensemble CV results. Best alpha = {BEST_ALPHA} ({BEST_TAG}).")

## 1. Overall metrics (5-fold CV mean ± std)

Standard classification metrics on each fold's **test** set; rows below are **means ± std** across **five** folds.

In [ ]:
def agg_row(name, key):
    a = ENS[key]["aggregate"]
    rows = []
    for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        mu = a.get(f"{m}_mean", np.nan)
        sd = a.get(f"{m}_std", np.nan)
        rows.append(f"{mu:.4f} ± {sd:.4f}")
    return {"Model": name, "Accuracy": rows[0], "Precision": rows[1], "Recall": rows[2], "F1": rows[3], "ROC-AUC": rows[4]}

overall = pd.DataFrame([
    agg_row("Random Forest", "random_forest"),
    agg_row("GraphSAGE (strongest individual graph model)", "graphsage_tuned"),
    agg_row("Ensemble 0.6·RF + 0.4·GNN (best overall CV classifier)", BEST_TAG),
])

display(overall.style.set_caption(
    "5-fold stratified CV: mean ± std across folds (not single-split test)."
).set_properties(**{"text-align": "left"}))

In [ ]:
def means_std(key):
    a = ENS[key]["aggregate"]
    out = {}
    for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        out[m] = (a.get(f"{m}_mean"), a.get(f"{m}_std"))
    return out

models = {
    "Random Forest": "random_forest",
    "GraphSAGE": "graphsage_tuned",
    "Ensemble": BEST_TAG,
}
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
labels = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]

x = np.arange(len(metrics))
w = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (mname, mkey) in enumerate(models.items()):
    ms = means_std(mkey)
    vals = [ms[k][0] for k in metrics]
    err = [ms[k][1] for k in metrics]
    ax.bar(x + (i - 1) * w, vals, w, yerr=err, capsize=3, label=mname)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("5-fold CV mean")
ax.set_title("Overall metrics (bars: mean; error bars: std across folds)")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 2. F1 and ROC-AUC emphasis (5-fold CV mean ± std)

These two metrics are often used together: **F1** for balanced precision/recall on the thresholded classifier; **ROC-AUC** for ranking quality.

In [ ]:
f1_roc = []
for label, key in [
    ("Random Forest", "random_forest"),
    ("GraphSAGE", "graphsage_tuned"),
    ("Ensemble", BEST_TAG),
]:
    a = ENS[key]["aggregate"]
    f1_roc.append({
        "Model": label,
        "F1 (mean ± std)": f"{a['f1_mean']:.4f} ± {a['f1_std']:.4f}",
        "ROC-AUC (mean ± std)": f"{a['roc_auc_mean']:.4f} ± {a['roc_auc_std']:.4f}",
    })
display(pd.DataFrame(f1_roc))

## 3. Precision@10 and Precision@20 (5-fold CV mean ± std)

In [ ]:
pk_rows = []
for label, key in [
    ("Random Forest", "random_forest"),
    ("GraphSAGE", "graphsage_tuned"),
    ("Ensemble", BEST_TAG),
]:
    a = ENS[key]["aggregate"]
    pk_rows.append({
        "Model": label,
        "P@10 (mean ± std)": f"{a['precision_at_10_mean']:.4f} ± {a['precision_at_10_std']:.4f}",
        "P@20 (mean ± std)": f"{a['precision_at_20_mean']:.4f} ± {a['precision_at_20_std']:.4f}",
    })
pk_df = pd.DataFrame(pk_rows)
display(pk_df)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3)
w = 0.35
for i, (col, off) in enumerate([("precision_at_10", -w / 2), ("precision_at_20", w / 2)]):
    means, stds = [], []
    for key in ["random_forest", "graphsage_tuned", BEST_TAG]:
        a = ENS[key]["aggregate"]
        means.append(a[f"{col}_mean"])
        stds.append(a[f"{col}_std"])
    ax.bar(x + off, means, w, yerr=stds, capsize=3, label=col.replace("precision_at", "P@"))
ax.set_xticks(x)
ax.set_xticklabels(["RF", "GraphSAGE", "Ensemble"])
ax.set_ylabel("5-fold CV mean")
ax.set_title("Precision@k (mean ± std across folds)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Per-node-type F1 (5-fold CV mean ± std)

In [ ]:
nts = ["PORT", "PLANT", "WAREHOUSE", "DC"]
rows = []
for nt in nts:
    row = {"Node type": nt}
    for label, key in [
        ("RF", "random_forest"),
        ("GraphSAGE", "graphsage_tuned"),
        ("Ensemble", BEST_TAG),
    ]:
        d = ENS[key]["per_node_type_f1"].get(nt, {})
        row[f"{label} F1"] = f"{d.get('f1_mean', 0):.4f} ± {d.get('f1_std', 0):.4f}"
    rows.append(row)
nt_df = pd.DataFrame(rows)
display(nt_df)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(nts))
w = 0.25
for i, (label, key) in enumerate([
    ("RF", "random_forest"),
    ("GraphSAGE", "graphsage_tuned"),
    ("Ensemble", BEST_TAG),
]):
    means = [ENS[key]["per_node_type_f1"].get(nt, {}).get("f1_mean", 0) for nt in nts]
    stds = [ENS[key]["per_node_type_f1"].get(nt, {}).get("f1_std", 0) for nt in nts]
    ax.bar(x + (i - 1) * w, means, w, yerr=stds, capsize=2, label=label)
ax.set_xticks(x)
ax.set_xticklabels(nts)
ax.set_ylabel("F1 (5-fold CV mean)")
ax.set_title("Per-node-type F1 (error bars: std across folds)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Ensemble weight sweep (alpha)

For each $\alpha \in \{0.3, 0.4, 0.5, 0.6, 0.7\}$: `ensemble_prob = α * gnn_prob + (1-α) * rf_prob`. The selected **final** ensemble uses **α = 0.4** (40% GraphSAGE, 60% Random Forest), which maximised **mean F1** across folds.

In [ ]:
alphas = [0.3, 0.4, 0.5, 0.6, 0.7]
f1_m, f1_s, auc_m, auc_s = [], [], [], []
for a in alphas:
    tag = f"ensemble_a{a:.1f}"
    ag = ENS[tag]["aggregate"]
    f1_m.append(ag["f1_mean"])
    f1_s.append(ag["f1_std"])
    auc_m.append(ag["roc_auc_mean"])
    auc_s.append(ag["roc_auc_std"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.errorbar(alphas, f1_m, yerr=f1_s, marker="o", capsize=4, label="F1 (mean ± std)")
ax1.set_xlabel("α (weight on GraphSAGE; RF weight = 1−α)")
ax1.set_ylabel("F1", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")
ax2 = ax1.twinx()
ax2.errorbar(alphas, auc_m, yerr=auc_s, marker="s", capsize=4, color="C1", label="ROC-AUC (mean ± std)")
ax2.set_ylabel("ROC-AUC", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")
ax1.axvline(BEST_ALPHA, color="gray", linestyle="--", alpha=0.8, label=f"best α = {BEST_ALPHA}")
ax1.set_title("Ensemble sweep: 5-fold CV means")
fig.tight_layout()
plt.show()

## 6. Interpretation

**Best overall cross-validated classifier:** The **ensemble** (60% Random Forest + 40% GraphSAGE probabilities, threshold 0.5) achieves the highest **5-fold CV mean F1** and **accuracy** among the models compared. It also has the highest **precision**, which matters when false alarms are costly. Reporting uses **mean ± std** across folds to show stability; the ensemble's F1 standard deviation is among the lowest, indicating consistent performance across different train/test partitions.

**Strongest individual graph model:** **GraphSAGE** has the highest **5-fold CV mean ROC-AUC** among **single** models. It is the appropriate reference when the question is how well a pure graph neural network ranks critical vs non-critical nodes without blending with Random Forest.

**Why combine models?** Random Forest excels at **non-linear patterns in tabular and structural features** but ignores edge-based message passing. GraphSAGE **propagates information along the supply-chain graph**. A linear blend of their positive-class probabilities combines complementary errors: the ensemble improves **PORT** and **PLANT** F1 especially, while **WAREHOUSE** remains difficult for all models (the ensemble can underperform GraphSAGE alone on that type in CV).

**Not single-split:** Figures and tables in this notebook are derived from **cross-validation**, not the 70/15/15 single split used for saving `best_model.pt` and early project notebooks.